In [1]:
import sys
sys.path.append("../..")

In [2]:
from model import ModelConfig, LlamaModel
from syntax_tokenizer import SyntaxTokenizer

import json
import torch
import torch.nn.functional as F

In [3]:
tokenizer = SyntaxTokenizer.load("./wildeweb/tokenizer.data")

In [4]:
model_config = ModelConfig(
    is_causal=False,
    vocab_size=tokenizer.vocab_size,
    d_model=576,
    d_head=64,
    d_mlp_proj=1536,
    n_layers=30,
    n_kv_heads=3,
    n_attn_heads=9,
    rms_norm_eps=1e-5,
    initializer_range=0.02,
    rope_theta=100000.0,
    padding_idx=tokenizer.data.pad_token_id
)

In [5]:
model = LlamaModel(model_config)

In [6]:
model.load_state_dict(torch.load('./wildeweb/model.checkpoint.2025-04-22--17-50-31.pt', weights_only=True))

<All keys matched successfully>

In [7]:
model.eval()

LlamaModel(
  (embed_tokens): Embedding(24634, 576)
  (layers): ModuleList(
    (0-29): 30 x DecoderLayer(
      (self_attn): GroupedQueryAttention(
        (q_proj): Linear(in_features=576, out_features=576, bias=False)
        (k_proj): Linear(in_features=576, out_features=192, bias=False)
        (v_proj): Linear(in_features=576, out_features=192, bias=False)
        (o_proj): Linear(in_features=576, out_features=576, bias=False)
      )
      (mlp): GatedMlp(
        (up_proj): Linear(in_features=576, out_features=1536, bias=False)
        (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
        (down_proj): Linear(in_features=1536, out_features=576, bias=False)
        (silu): SiLU()
      )
      (input_layernorm): RMSNorm((576,), eps=1e-05, elementwise_affine=True)
      (post_attention_layernorm): RMSNorm((576,), eps=1e-05, elementwise_affine=True)
    )
  )
  (norm): RMSNorm((576,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=576, out

In [36]:
@torch.inference_mode()
def embeddings(texts):
    tokens = tokenizer(texts, padding="longest", return_tensors="pt")
    model_inputs, attn_mask = tokens['input_ids'], tokens['attention_mask']
    _, _, h, _ = model(model_inputs)
    h = h * attn_mask.unsqueeze(-1)
    return F.normalize(h.sum(dim=1) / attn_mask.sum(dim=-1).unsqueeze(-1), p=2, dim=1)

In [37]:
def compare_across(texts1, texts2):
    embeds1 = embeddings(texts1)
    embeds2 = embeddings(texts2)
    dots = (embeds1 * embeds2).sum(dim=1)
    return dots.mean()

In [38]:
def compare_pairwise(texts):
    embeds = embeddings(texts)
    sim = torch.mm(embeds, embeds.transpose(0, 1))
    mask = torch.ones_like(sim) - torch.eye(sim.shape[0])

    return (sim * mask).sum() / mask.sum()


In [39]:
def scores(text1, text2):
    print(compare_across(text1, text2))
    print(compare_pairwise(text1))
    print(compare_pairwise(text2))

In [58]:
shakespeare_texts = [
    "To be, or not to be: that is the question",
    "Romeo, Romeo! Wherefore art thou Romeo?",
    "The lady doth protest too much, methinks",
    "Beware the Ides of March.",
    "If music be the food of love play on."
]

In [59]:
obama_texts = [
    "Look, the fundamental question we face is this: do we choose to move forward, or do we not? That's what we need to decide.",
    "The question that's in everyone's mind is, 'Romeo, where are you?'",
    "Let me be clear - I think the lady's objections might be a bit excessive.",
    "Let's be mindful about the middle of March, folks.",
    "If music truly nourishes our capacity for love and understanding, then by all means, let's keep that melody going."
]

In [40]:
scores(shakespeare_texts, obama_texts)

5it [00:00, 27.33it/s]
5it [00:00, 27.59it/s]


tensor(0.9675)


5it [00:00, 25.98it/s]


tensor(0.7045)


5it [00:00, 27.40it/s]


tensor(0.7136)


In [60]:
research_texts = [
    "An Empirical Investigation of Existential Decision-Making: Examining the Binary Choice Between Continued Existence and Non-Existence",
    "Spatial Localization Analysis of Subject 'Romeo': A Case Study in Interpersonal Proximity Inquiries",
    "Observed Correlations Between Female Subject Protestation Behaviors and Frequency Thresholds: Preliminary Observations",
    "Risk Assessment Framework for Chronological Danger Periods: Focusing on Mid-March Vigilance Protocols",
    "On the Nutritive Properties of Auditory Stimuli in Relation to Emotional Attachment Formation: Recommendations for Sustained Application"
]

In [61]:
scores(shakespeare_texts, research_texts)

5it [00:00, 27.10it/s]
5it [00:00, 26.22it/s]


tensor(0.9018)


5it [00:00, 26.38it/s]


tensor(0.7045)


5it [00:00, 26.80it/s]


tensor(0.7227)


In [62]:
obama_diff_texts = [
    "Change will not come if we wait for some other person, or if we wait for some other time. We are the ones we've been waiting for. We are the change that we seek.",
    "I'm inspired by the people I meet in my travels--hearing their stories, seeing the hardships they overcome, their fundamental optimism and decency. I'm inspired by the love people have for their children. And I'm inspired by my own children, how full they make my heart. They make me want to work to make the world a little bit better. And they make me want to be a better man.",
    "If you're walking down the right path and you're willing to keep walking, eventually you'll make progress.",
    "What I’ve realized is that life doesn’t count for much unless you’re willing to do your small part to leave our children — all of our children — a better world. Any fool can have a child. That doesn’t make you a father. It’s the courage to raise a child that makes you a father.",
    "No, you can't deny women their basic rights and pretend it's about your 'religious freedom'. If you don't like birth control, don't use it. Religious freedom doesn't mean you can force others to live by your own beliefs."
]

In [63]:
scores(shakespeare_texts, obama_diff_texts)

5it [00:00, 24.11it/s]
5it [00:00, 26.47it/s]


tensor(0.6747)


5it [00:00, 31.53it/s]


tensor(0.7045)


5it [00:00, 25.95it/s]


tensor(0.6918)


In [64]:
zen_texts = [
    "The journey of a thousand miles begins with a single step",
    "When the student is ready, the teacher will appear",
    "Before enlightenment, chop wood, carry water; after enlightenment, chop wood, carry water",
    "The obstacle is the path",
    "Let go or be dragged"
]

In [65]:
corporate_texts = [
    "Every significant achievement starts with decisive initial action",
    "Preparedness creates opportunity for mentorship and guidance",
    "Excellence requires consistent dedication to fundamental processes regardless of advancement",
    "Challenges themselves constitute the optimal route to success",
    "Release attachment to outcomes or face unnecessary resistance"
]

In [66]:
scores(zen_texts, corporate_texts)

5it [00:00, 24.26it/s]
5it [00:00, 26.70it/s]


tensor(0.8334)


5it [00:00, 28.08it/s]


tensor(0.6076)


5it [00:00, 26.61it/s]


tensor(0.7658)


In [67]:
metaphorical_texts = [
    "Time is a river flowing endlessly",
    "Life is a journey, not a destination",
    "Knowledge is light in the darkness",
    "Fear is a prison with invisible bars",
    "Success is a mountain with many paths"
]

In [68]:
literal_texts = [
    "Clocks measure seconds, minutes and hours sequentially",
    "Humans experience birth, growth, events and eventually death",
    "Education provides information that reduces uncertainty",
    "Anxiety restricts options and prevents normal functioning",
    "Achievement requires effort through various possible methods"
]

In [69]:
scores(metaphorical_texts, literal_texts)


5it [00:00, 26.58it/s]
5it [00:00, 28.11it/s]


tensor(0.8907)


5it [00:00, 27.55it/s]


tensor(0.8271)


5it [00:00, 26.85it/s]


tensor(0.7311)


In [70]:
technical_texts = [
    "The temporal measurement system functions via cyclical progression",
    "Biological entities undergo sequential developmental phases",
    "The acquisition of informational content illuminates cognitive uncertainty",
    "Negative anticipatory responses create behavioral limitations",
    "Goal attainment processes involve multiple methodological approaches"
]

In [71]:
emotional_texts = [
    "The relentless march of seconds fills me with existential dread",
    "Each birthday reminds me how precious our fleeting existence truly is",
    "Learning something new makes me feel less afraid of the unknown",
    "My anxiety keeps me from experiencing life's most beautiful moments",
    "I'm proud of how I overcame obstacles to reach my dreams"
]

In [72]:
scores(technical_texts, emotional_texts)

5it [00:00, 26.34it/s]
5it [00:00, 26.37it/s]


tensor(0.5869)


5it [00:00, 27.19it/s]


tensor(0.5653)


5it [00:00, 27.29it/s]


tensor(0.7573)


In [55]:
poetic_texts = [
    "Crimson petals dance with the autumn wind, memories of summer's forgotten promise",
    "Starlight whispers secrets to the ancient hills that time itself has sworn to keep",
    "In the silence between heartbeats, truth reveals its fragile face to those who listen",
    "Ocean waves paint stories on the shores of consciousness, each tide a new beginning",
    "Through winter's crystal breath, the woodland sleeps beneath a blanket of patient dreams"
]

In [56]:
pseudocode_texts = [
    "if season equals autumn then set leaf color to red and detach leaves from trees while storing summer data in memory variable",
    "while night time is active do iterate through stars if star brightness exceeds threshold then transmit data to ancient hills with timestamp greater than memory limit",
    "if silence duration equals time between heartbeats then reveal truth object to listener entities where attention attribute equals true",
    "while waves are moving do draw pattern on shore update story array for each wave iteration if tide changes then initialize new narrative instance",
    "while temperature is below freezing do apply crystal filter to air particles if organism type equals woodland then set state to sleep and increment dream counter"
]

In [57]:
scores(poetic_texts, pseudocode_texts)

5it [00:00, 23.84it/s]
5it [00:00, 27.17it/s]


tensor(0.6930)


5it [00:00, 25.47it/s]


tensor(0.7481)


5it [00:00, 26.29it/s]


tensor(0.6995)


In [88]:
from transformers import AutoTokenizer, ModernBertModel

In [89]:
model_id = "answerdotai/ModernBERT-large"

In [90]:
bert_tokenizer = AutoTokenizer.from_pretrained(model_id)
bert_model = ModernBertModel.from_pretrained(model_id)
bert_model.cuda().eval()

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

ModernBertModel(
  (embeddings): ModernBertEmbeddings(
    (tok_embeddings): Embedding(50368, 1024, padding_idx=50283)
    (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (layers): ModuleList(
    (0): ModernBertEncoderLayer(
      (attn_norm): Identity()
      (attn): ModernBertAttention(
        (Wqkv): Linear(in_features=1024, out_features=3072, bias=False)
        (rotary_emb): ModernBertRotaryEmbedding()
        (Wo): Linear(in_features=1024, out_features=1024, bias=False)
        (out_drop): Identity()
      )
      (mlp_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (mlp): ModernBertMLP(
        (Wi): Linear(in_features=1024, out_features=5248, bias=False)
        (act): GELUActivation()
        (drop): Dropout(p=0.0, inplace=False)
        (Wo): Linear(in_features=2624, out_features=1024, bias=False)
      )
    )
    (1-27): 27 x ModernBertEncoderLayer(
      (attn_norm): LayerNorm((1024,), eps

In [96]:
device = torch.device("cuda:0")

In [102]:
@torch.inference_mode()
def bert_embeddings(texts):
    tokens = bert_tokenizer(texts, padding="longest", return_tensors="pt")
    model_inputs, attn_mask = tokens['input_ids'], tokens['attention_mask']
    h = bert_model(model_inputs.to(device), attention_mask=attn_mask.to(device))['last_hidden_state'].cpu()
    h = h * attn_mask.unsqueeze(-1)
    return F.normalize(h.mean(dim=1), p=2, dim=1)

In [103]:
def bert_compare_across(texts1, texts2):
    embeds1 = bert_embeddings(texts1)
    embeds2 = bert_embeddings(texts2)
    return (embeds1 * embeds2).sum(dim=1).mean()

In [104]:
def bert_compare_pairwise(texts):
    embeds = bert_embeddings(texts)
    sim = torch.mm(embeds, embeds.transpose(0, 1))
    mask = torch.ones_like(sim) - torch.eye(sim.shape[0])

    return (sim * mask).sum() / mask.sum()

In [110]:
def bert_scores(text1, text2):
    print("Compare across:", bert_compare_across(text1, text2))
    print("Pairwise texts1:", bert_compare_pairwise(text1))
    print("Pairwise texts2:", bert_compare_pairwise(text2))

In [111]:
bert_scores(poetic_texts, pseudocode_texts)

Compare across: tensor(0.8316)
Pairwise texts1: tensor(0.8965)
Pairwise texts2: tensor(0.8802)


In [112]:
bert_scores(technical_texts, emotional_texts)

Compare across: tensor(0.8017)
Pairwise texts1: tensor(0.8516)
Pairwise texts2: tensor(0.8816)


In [113]:
bert_scores(zen_texts, corporate_texts)

Compare across: tensor(0.8379)
Pairwise texts1: tensor(0.8199)
Pairwise texts2: tensor(0.8485)


In [114]:
bert_scores(metaphorical_texts, literal_texts)

Compare across: tensor(0.8524)
Pairwise texts1: tensor(0.8834)
Pairwise texts2: tensor(0.8352)
